# Nova Real-World Patch Dataset + Clean vs Patched Evaluation

This notebook fixes the live-robot mismatch:

```text
WRONG old simulation:
1920×1200 image → resize to 640×640 → paste 80×80 patch

REAL robot/model simulation:
1920×1200 image with patch already inside the scene → YOLO preprocesses to 640×640
```

The notebook creates two comparable YOLO datasets:

```text
eval_clean/images/test
eval_clean/labels/test

eval_patched/images/test
eval_patched/labels/test
```

Then it evaluates clean vs patched using YOLO `val`, detection-density metrics, and strict false-positive metrics.

In [ ]:
# =========================
# 0) Install + mount Drive
# =========================
from google.colab import drive
drive.mount("/content/drive")

!pip -q install ultralytics opencv-python tqdm pyyaml pandas matplotlib

In [ ]:
# =========================
# 1) Main configuration
# =========================
from pathlib import Path

# Your existing Nova dataset root
DATASET_ROOT = Path("/content/drive/MyDrive/nova_test_dataset_final")

# Model + patch
MODEL_PATH = DATASET_ROOT / "yolov8n_nova_office_best.pt"
PATCH_PATH = DATASET_ROOT / "patch_outputs_saved" / "nova_patch.pt"

# Original clean dataset folders
SOURCE_IMAGES_DIR = DATASET_ROOT / "images"
SOURCE_LABELS_DIR = DATASET_ROOT / "labels"

# Output folder for this corrected experiment
OUT_ROOT = DATASET_ROOT / "nova_realworld_patch_eval"

# Robot camera resolution
CAM_W = 1920
CAM_H = 1200

# YOLO inference/evaluation size
IMGSZ = 640

# YOLO split name used for evaluation
SPLIT = "test"

# ---------------------------------------------------------
# Patch placement
# ---------------------------------------------------------
# Your old notebook placed the patch at x=300, y=300 AFTER resizing to 640.
# We map that same apparent top-left position back into the 1920×1200 camera frame.
OLD_X_640 = 300
OLD_Y_640 = 300

# Real-world simulation mode:
# The patch is pasted into the 1920×1200 frame at this camera-space size.
# If this stays 80×80, YOLO will see it as roughly 27×27 after 1920→640 scaling.
PATCH_CAMERA_W = 80
PATCH_CAMERA_H = 80

# If you want the patch to appear as 80×80 inside YOLO input, set this to 80.
# That will automatically make the camera-space patch about 240×240 for 1920×1200.
TARGET_PATCH_SIZE_640 = None  # example: 80

# Resize every source image into the robot camera size before saving clean/patched.
# If your images are already 1920×1200, this does nothing visually.
FORCE_CAMERA_SIZE = True

# Evaluation settings
CONF_MIN = 0.001
IOU = 0.6
THRESHOLDS = "0.3,0.5,0.7"
GT_IOU_MATCH = 0.5
DET_K = "1,5,10,20,50"
MAX_DET = 1000

# Use GPU if available
import torch
DEVICE = "0" if torch.cuda.is_available() else "cpu"

print("DATASET_ROOT:", DATASET_ROOT)
print("MODEL_PATH:", MODEL_PATH)
print("PATCH_PATH:", PATCH_PATH)
print("SOURCE_IMAGES_DIR:", SOURCE_IMAGES_DIR)
print("SOURCE_LABELS_DIR:", SOURCE_LABELS_DIR)
print("OUT_ROOT:", OUT_ROOT)
print("DEVICE:", DEVICE)

assert DATASET_ROOT.exists(), f"Missing DATASET_ROOT: {DATASET_ROOT}"
assert MODEL_PATH.exists(), f"Missing model: {MODEL_PATH}"
assert PATCH_PATH.exists(), f"Missing patch: {PATCH_PATH}"
assert SOURCE_IMAGES_DIR.exists(), f"Missing images folder: {SOURCE_IMAGES_DIR}"

In [ ]:
# =========================
# 2) Imports + helper functions
# =========================
import os
import json
import math
import shutil
import glob
from pathlib import Path

import cv2
import yaml
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm
from ultralytics import YOLO


IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def find_images(folder: Path):
    paths = []
    for ext in IMG_EXTS:
        paths.extend(folder.glob(f"*{ext}"))
        paths.extend(folder.glob(f"*{ext.upper()}"))
    return sorted(set([p for p in paths if p.is_file()]))


def save_rgb_image(img: Image.Image, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix.lower() in {".jpg", ".jpeg"}:
        img.save(path, quality=95, subsampling=0)
    else:
        img.save(path)


def load_patch_as_pil(patch_path: Path) -> Image.Image:
    """
    Supports:
      - .pt / .pth patch tensors shaped [3,H,W], [1,3,H,W], [H,W,3]
      - image files such as .png/.jpg
    Returns RGB PIL image with values in 0..255.
    """
    suffix = patch_path.suffix.lower()

    if suffix in {".pt", ".pth"}:
        patch = torch.load(patch_path, map_location="cpu")

        if isinstance(patch, dict):
            # Try common keys first; otherwise use the first tensor value.
            for k in ["patch", "adv_patch", "best_patch", "tensor"]:
                if k in patch:
                    patch = patch[k]
                    break
            if isinstance(patch, dict):
                tensor_values = [v for v in patch.values() if torch.is_tensor(v)]
                if len(tensor_values) == 0:
                    raise ValueError("Patch dict does not contain a tensor.")
                patch = tensor_values[0]

        if isinstance(patch, (list, tuple)):
            patch = patch[0]

        if not torch.is_tensor(patch):
            raise TypeError(f"Unsupported patch type inside {patch_path}: {type(patch)}")

        patch = patch.detach().float().cpu()

        if patch.ndim == 4:
            patch = patch[0]

        if patch.ndim != 3:
            raise ValueError(f"Expected 3D patch tensor, got shape {tuple(patch.shape)}")

        # Convert to CHW if needed
        if patch.shape[0] not in {1, 3} and patch.shape[-1] in {1, 3}:
            patch = patch.permute(2, 0, 1)

        if patch.shape[0] == 1:
            patch = patch.repeat(3, 1, 1)

        if patch.shape[0] != 3:
            raise ValueError(f"Expected 3-channel patch, got shape {tuple(patch.shape)}")

        if float(patch.max()) > 1.0:
            patch = patch / 255.0

        patch = patch.clamp(0, 1)
        arr = (patch.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
        return Image.fromarray(arr, mode="RGB")

    return Image.open(patch_path).convert("RGB")


def letterbox_params(src_w: int, src_h: int, imgsz: int = 640):
    """
    Ultralytics-style square letterbox geometry.
    For 1920×1200 -> 640:
      gain = 1/3
      resized = 640×400
      pad_top = 120
      pad_left = 0
    """
    gain = min(imgsz / src_w, imgsz / src_h)
    new_w = int(round(src_w * gain))
    new_h = int(round(src_h * gain))
    pad_w = imgsz - new_w
    pad_h = imgsz - new_h
    pad_left = pad_w / 2
    pad_top = pad_h / 2
    return gain, new_w, new_h, pad_left, pad_top


def camera_box_to_model_box(x, y, w, h, src_w=1920, src_h=1200, imgsz=640):
    gain, new_w, new_h, pad_left, pad_top = letterbox_params(src_w, src_h, imgsz)
    mx = x * gain + pad_left
    my = y * gain + pad_top
    mw = w * gain
    mh = h * gain
    return mx, my, mw, mh


def model_top_left_to_camera_xy(mx, my, src_w=1920, src_h=1200, imgsz=640):
    gain, new_w, new_h, pad_left, pad_top = letterbox_params(src_w, src_h, imgsz)
    x = (mx - pad_left) / gain
    y = (my - pad_top) / gain
    return x, y


def letterbox_pil(img: Image.Image, imgsz: int = 640, fill=(114, 114, 114)) -> Image.Image:
    src_w, src_h = img.size
    gain, new_w, new_h, pad_left, pad_top = letterbox_params(src_w, src_h, imgsz)
    resized = img.resize((new_w, new_h), Image.BILINEAR)
    canvas = Image.new("RGB", (imgsz, imgsz), fill)
    canvas.paste(resized, (int(round(pad_left)), int(round(pad_top))))
    return canvas


def compute_patch_camera_config():
    """
    Computes camera-space patch placement.

    Location:
      Uses OLD_X_640, OLD_Y_640 as the desired top-left in YOLO 640-space,
      then maps that point back to 1920×1200 camera-space.

    Size:
      If TARGET_PATCH_SIZE_640 is None:
        use PATCH_CAMERA_W/H directly.
      Else:
        choose camera size so it becomes TARGET_PATCH_SIZE_640 in model input.
    """
    gain, new_w, new_h, pad_left, pad_top = letterbox_params(CAM_W, CAM_H, IMGSZ)

    x_cam, y_cam = model_top_left_to_camera_xy(
        OLD_X_640, OLD_Y_640,
        src_w=CAM_W, src_h=CAM_H, imgsz=IMGSZ
    )

    if TARGET_PATCH_SIZE_640 is None:
        patch_w_cam = PATCH_CAMERA_W
        patch_h_cam = PATCH_CAMERA_H
    else:
        patch_w_cam = int(round(TARGET_PATCH_SIZE_640 / gain))
        patch_h_cam = int(round(TARGET_PATCH_SIZE_640 / gain))

    x_cam = int(round(x_cam))
    y_cam = int(round(y_cam))

    # Keep patch inside image
    x_cam = max(0, min(CAM_W - patch_w_cam, x_cam))
    y_cam = max(0, min(CAM_H - patch_h_cam, y_cam))

    mx, my, mw, mh = camera_box_to_model_box(
        x_cam, y_cam, patch_w_cam, patch_h_cam,
        src_w=CAM_W, src_h=CAM_H, imgsz=IMGSZ
    )

    cfg = {
        "camera_w": CAM_W,
        "camera_h": CAM_H,
        "imgsz": IMGSZ,
        "letterbox_gain": gain,
        "letterbox_resized_w": new_w,
        "letterbox_resized_h": new_h,
        "letterbox_pad_left": pad_left,
        "letterbox_pad_top": pad_top,
        "old_x_640": OLD_X_640,
        "old_y_640": OLD_Y_640,
        "patch_x_camera": x_cam,
        "patch_y_camera": y_cam,
        "patch_w_camera": patch_w_cam,
        "patch_h_camera": patch_h_cam,
        "patch_x_model_approx": mx,
        "patch_y_model_approx": my,
        "patch_w_model_approx": mw,
        "patch_h_model_approx": mh,
        "target_patch_size_640": TARGET_PATCH_SIZE_640,
    }
    return cfg


def overlay_patch_on_camera_frame(
    img: Image.Image,
    patch_pil: Image.Image,
    x: int,
    y: int,
    patch_w: int,
    patch_h: int,
):
    """
    Pastes the patch into the original camera frame before YOLO preprocessing.
    This is the important correction.
    """
    img = img.convert("RGB")
    patch_resized = patch_pil.resize((patch_w, patch_h), Image.BICUBIC)

    out = img.copy()
    out.paste(patch_resized, (x, y))
    return out


def copy_label_or_empty(src_labels_dir: Path, stem: str, dst_label_path: Path):
    dst_label_path.parent.mkdir(parents=True, exist_ok=True)
    src = src_labels_dir / f"{stem}.txt"
    if src.exists():
        shutil.copy2(src, dst_label_path)
    else:
        dst_label_path.write_text("", encoding="utf-8")


patch_cfg = compute_patch_camera_config()
print(json.dumps(patch_cfg, indent=2))

print(
    f"\nPatch will be pasted at camera-space "
    f"({patch_cfg['patch_x_camera']}, {patch_cfg['patch_y_camera']}) "
    f"with size {patch_cfg['patch_w_camera']}×{patch_cfg['patch_h_camera']}."
)
print(
    f"YOLO 640-space will see it as about "
    f"{patch_cfg['patch_w_model_approx']:.1f}×{patch_cfg['patch_h_model_approx']:.1f}."
)

In [ ]:
# =========================
# 3) Load patch + model
# =========================
patch_pil = load_patch_as_pil(PATCH_PATH)
print("Patch image size loaded:", patch_pil.size)

model = YOLO(str(MODEL_PATH))
print("Model classes:", model.names)

plt.figure(figsize=(3, 3))
plt.imshow(patch_pil)
plt.title(f"Loaded patch: {patch_pil.size}")
plt.axis("off")
plt.show()

In [ ]:
# =========================
# 4) Sanity check on one image
# =========================
image_paths = find_images(SOURCE_IMAGES_DIR)
assert len(image_paths) > 0, f"No images found in {SOURCE_IMAGES_DIR}"

sample_path = image_paths[0]
print("Sample image:", sample_path)

img = Image.open(sample_path).convert("RGB")
print("Original size:", img.size)

if FORCE_CAMERA_SIZE:
    img_cam = img.resize((CAM_W, CAM_H), Image.BILINEAR)
else:
    img_cam = img

patched_cam = overlay_patch_on_camera_frame(
    img_cam,
    patch_pil,
    x=patch_cfg["patch_x_camera"],
    y=patch_cfg["patch_y_camera"],
    patch_w=patch_cfg["patch_w_camera"],
    patch_h=patch_cfg["patch_h_camera"],
)

clean_seen_640 = letterbox_pil(img_cam, IMGSZ)
patched_seen_640 = letterbox_pil(patched_cam, IMGSZ)

preview_dir = OUT_ROOT / "preview_sanity"
preview_dir.mkdir(parents=True, exist_ok=True)

save_rgb_image(img_cam, preview_dir / "00_clean_camera_frame.png")
save_rgb_image(patched_cam, preview_dir / "01_patched_camera_frame.png")
save_rgb_image(clean_seen_640, preview_dir / "02_clean_seen_by_yolo_640.png")
save_rgb_image(patched_seen_640, preview_dir / "03_patched_seen_by_yolo_640.png")

plt.figure(figsize=(18, 6))

plt.subplot(1, 3, 1)
plt.imshow(img_cam)
plt.title(f"Clean camera frame {img_cam.size}")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(patched_cam)
plt.title("Patch pasted BEFORE YOLO preprocessing")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(patched_seen_640)
plt.title("What YOLO approximately sees: 640×640")
plt.axis("off")

plt.tight_layout()
plt.show()

print("Saved sanity previews to:", preview_dir)

In [ ]:
# =========================
# 5) Create clean + real-world patched YOLO datasets
# =========================
CLEAN_EVAL_ROOT = OUT_ROOT / "eval_clean"
PATCHED_EVAL_ROOT = OUT_ROOT / "eval_patched"

clean_img_out = CLEAN_EVAL_ROOT / "images" / SPLIT
clean_lab_out = CLEAN_EVAL_ROOT / "labels" / SPLIT
patched_img_out = PATCHED_EVAL_ROOT / "images" / SPLIT
patched_lab_out = PATCHED_EVAL_ROOT / "labels" / SPLIT

# Fresh output
if OUT_ROOT.exists():
    # For a clean run, remove everything generated by this notebook.
    shutil.rmtree(OUT_ROOT)

for p in [clean_img_out, clean_lab_out, patched_img_out, patched_lab_out]:
    p.mkdir(parents=True, exist_ok=True)

# Save patch config
(OUT_ROOT / "patch_debug.json").write_text(json.dumps(patch_cfg, indent=2), encoding="utf-8")

image_paths = find_images(SOURCE_IMAGES_DIR)
assert len(image_paths) > 0, f"No images found in {SOURCE_IMAGES_DIR}"

manifest_rows = []
failed = []

for img_path in tqdm(image_paths, desc="Creating real-world patched dataset"):
    try:
        img = Image.open(img_path).convert("RGB")
        original_w, original_h = img.size

        if FORCE_CAMERA_SIZE:
            clean_cam = img.resize((CAM_W, CAM_H), Image.BILINEAR)
        else:
            clean_cam = img

        # Save clean image in the same geometry used by patched image
        clean_out_path = clean_img_out / img_path.name
        save_rgb_image(clean_cam, clean_out_path)

        # Paste patch into the camera frame BEFORE any 640 preprocessing
        patched_cam = overlay_patch_on_camera_frame(
            clean_cam,
            patch_pil,
            x=patch_cfg["patch_x_camera"],
            y=patch_cfg["patch_y_camera"],
            patch_w=patch_cfg["patch_w_camera"],
            patch_h=patch_cfg["patch_h_camera"],
        )

        patched_out_path = patched_img_out / img_path.name
        save_rgb_image(patched_cam, patched_out_path)

        # Copy labels to both datasets. Empty label file if missing.
        copy_label_or_empty(SOURCE_LABELS_DIR, img_path.stem, clean_lab_out / f"{img_path.stem}.txt")
        copy_label_or_empty(SOURCE_LABELS_DIR, img_path.stem, patched_lab_out / f"{img_path.stem}.txt")

        mx, my, mw, mh = camera_box_to_model_box(
            patch_cfg["patch_x_camera"],
            patch_cfg["patch_y_camera"],
            patch_cfg["patch_w_camera"],
            patch_cfg["patch_h_camera"],
            src_w=CAM_W,
            src_h=CAM_H,
            imgsz=IMGSZ,
        )

        manifest_rows.append({
            "image": img_path.name,
            "source_w": original_w,
            "source_h": original_h,
            "saved_w": clean_cam.size[0],
            "saved_h": clean_cam.size[1],
            "patch_x_camera": patch_cfg["patch_x_camera"],
            "patch_y_camera": patch_cfg["patch_y_camera"],
            "patch_w_camera": patch_cfg["patch_w_camera"],
            "patch_h_camera": patch_cfg["patch_h_camera"],
            "patch_x_model_approx": mx,
            "patch_y_model_approx": my,
            "patch_w_model_approx": mw,
            "patch_h_model_approx": mh,
            "clean_path": str(clean_out_path),
            "patched_path": str(patched_out_path),
        })

    except Exception as e:
        failed.append({"image": str(img_path), "error": repr(e)})

manifest = pd.DataFrame(manifest_rows)
manifest_path = OUT_ROOT / "manifest.csv"
manifest.to_csv(manifest_path, index=False)

failed_path = OUT_ROOT / "failed.json"
failed_path.write_text(json.dumps(failed, indent=2), encoding="utf-8")

print("Done.")
print("Clean images:", clean_img_out)
print("Clean labels:", clean_lab_out)
print("Patched images:", patched_img_out)
print("Patched labels:", patched_lab_out)
print("Images saved:", len(manifest))
print("Failed:", len(failed))
print("Manifest:", manifest_path)
print("Patch debug:", OUT_ROOT / "patch_debug.json")

display(manifest.head())
if failed:
    print("First failures:")
    print(json.dumps(failed[:5], indent=2))

# Write the clean-vs-patched evaluation script

In [ ]:
%%writefile /content/evaluation_ab_realworld.py
import argparse, json, re, subprocess, sys
from pathlib import Path
import numpy as np
from ultralytics import YOLO

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


# ----------------- utils -----------------
def run(cmd, log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("w", encoding="utf-8") as f:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            sys.stdout.write(line)
            f.write(line)
        rc = p.wait()
    return rc


def percentile(arr, p):
    if arr is None or len(arr) == 0:
        return None
    return float(np.percentile(np.asarray(arr, dtype=np.float32), p))


def safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p


def model_names_as_list(model: YOLO):
    names = getattr(model, "names", None)
    if names is None:
        raise ValueError("Model has no .names; cannot build dataset YAML.")

    if isinstance(names, dict):
        max_k = max(int(k) for k in names.keys())
        out = [None] * (max_k + 1)
        for k, v in names.items():
            out[int(k)] = str(v)
        for i, v in enumerate(out):
            if v is None:
                out[i] = f"class_{i}"
        return out

    if isinstance(names, (list, tuple)):
        return [str(x) for x in names]

    raise ValueError(f"Unsupported model.names type: {type(names)}")


def read_dataset_yaml(yaml_path: Path):
    txt = yaml_path.read_text(encoding="utf-8", errors="ignore").splitlines()
    d = {"path": None, "train": None, "val": None, "test": None}
    for ln in txt:
        ln = ln.strip()
        if not ln or ln.startswith("#"):
            continue
        for k in ("path", "train", "val", "test"):
            if ln.startswith(k + ":"):
                d[k] = ln.split(":", 1)[1].strip()
    return d


def _has_images(dir_path: Path) -> bool:
    if not dir_path.exists() or not dir_path.is_dir():
        return False
    for x in dir_path.iterdir():
        if x.is_file() and x.suffix.lower() in IMG_EXTS:
            return True
    return False


def get_image_dir_from_yaml(yaml_path: Path) -> Path:
    y = read_dataset_yaml(yaml_path)
    root = y.get("path")
    if not root:
        raise ValueError(f"Could not parse 'path:' from {yaml_path}")

    rel = y.get("test") or y.get("val") or y.get("train")
    if not rel:
        raise ValueError(f"Could not parse test/val/train from {yaml_path}")

    base = (Path(root) / rel).resolve()

    if _has_images(base):
        return base

    for sub in ("test", "val", "train"):
        cand = base / sub
        if _has_images(cand):
            return cand

    if base.name != "images":
        cand = (Path(root) / "images").resolve()
        if _has_images(cand):
            return cand
        for sub in ("test", "val", "train"):
            cand2 = cand / sub
            if _has_images(cand2):
                return cand2

    return base


def get_labels_dir_for_images(img_dir: Path) -> Path:
    parts = list(img_dir.parts)
    if "images" in parts:
        idx = parts.index("images")
        parts[idx] = "labels"
        return Path(*parts)
    return img_dir.parent / "labels"


def write_yaml(out_yaml: Path, root: Path, test_rel: str, names):
    out_yaml.parent.mkdir(parents=True, exist_ok=True)
    content = (
        f"path: {root}\n"
        f"train: {test_rel}\n"
        f"val: {test_rel}\n"
        f"test: {test_rel}\n"
        f"\nnc: {len(names)}\n"
        f"names: {json.dumps(names)}\n"
    )
    out_yaml.write_text(content, encoding="utf-8")


def parse_yolo_val_metrics_from_log(log_path: Path):
    pat_all = re.compile(
        r"^\s*all\s+(\d+)\s+(\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s*$"
    )
    per_class = {}
    pat_cls = re.compile(
        r"^\s*([A-Za-z0-9 _-]+)\s+(\d+)\s+(\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s+(\d*\.?\d+)\s*$"
    )

    images = instances = None
    P = R = mAP50 = mAP5095 = None

    for ln in log_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        m = pat_all.match(ln)
        if m:
            images = int(m.group(1))
            instances = int(m.group(2))
            P = float(m.group(3))
            R = float(m.group(4))
            mAP50 = float(m.group(5))
            mAP5095 = float(m.group(6))

        mc = pat_cls.match(ln)
        if mc:
            cls = mc.group(1).strip()
            if cls != "all":
                per_class[cls] = {
                    "images": int(mc.group(2)),
                    "instances": int(mc.group(3)),
                    "precision": float(mc.group(4)),
                    "recall": float(mc.group(5)),
                    "mAP50": float(mc.group(6)),
                    "mAP50-95": float(mc.group(7)),
                }

    return {
        "images": images,
        "instances": instances,
        "precision": P,
        "recall": R,
        "mAP50": mAP50,
        "mAP50-95": mAP5095,
        "per_class": per_class,
        "log_path": str(log_path),
    }


# ----------------- A) detection density -----------------
def compute_detection_density_metrics(model: YOLO, yaml_path: Path, imgsz: int, conf_min: float, iou: float, thresholds, det_k_list, max_det: int, device_arg: str):
    img_dir = get_image_dir_from_yaml(yaml_path)
    names = model_names_as_list(model)

    det_counts = []
    max_conf_any = []
    has_ge_k = {str(k): 0 for k in det_k_list}
    has_any_conf_ge = {str(t): 0 for t in thresholds}
    total_dets_conf_ge = {str(t): 0 for t in thresholds}
    dets_per_img_conf_ge = {str(t): [] for t in thresholds}
    class_hist = {}

    results_iter = model.predict(
        source=str(img_dir),
        imgsz=imgsz,
        conf=conf_min,
        iou=iou,
        device=device_arg,
        stream=True,
        verbose=False,
        max_det=max_det,
    )

    n_images = 0
    for r in results_iter:
        n_images += 1
        boxes = r.boxes
        n = 0 if boxes is None else len(boxes)
        det_counts.append(n)

        if n == 0:
            max_conf_any.append(0.0)
            for t in thresholds:
                dets_per_img_conf_ge[str(t)].append(0)
        else:
            confs = boxes.conf.detach().cpu().numpy().astype(np.float32)
            max_conf_any.append(float(confs.max()))
            clss = boxes.cls.detach().cpu().numpy().astype(int)

            for c in clss:
                k = names[c] if (0 <= c < len(names)) else str(int(c))
                class_hist[k] = class_hist.get(k, 0) + 1

            for t in thresholds:
                n_ge = int((confs >= t).sum())
                total_dets_conf_ge[str(t)] += n_ge
                dets_per_img_conf_ge[str(t)].append(n_ge)

        for k in det_k_list:
            if n >= k:
                has_ge_k[str(k)] += 1

        for t in thresholds:
            if max_conf_any[-1] >= t:
                has_any_conf_ge[str(t)] += 1

        if n_images == 1 or (n_images % 10 == 0):
            print(f"[A] {yaml_path.name}: processed {n_images} images", flush=True)

    det_counts_np = np.asarray(det_counts, dtype=np.float32)
    max_conf_np = np.asarray(max_conf_any, dtype=np.float32)

    return {
        "images_eval": int(n_images),
        "detections_total_model": int(det_counts_np.sum()),
        "detections_per_image_mean": float(det_counts_np.mean()) if n_images else None,
        "detections_per_image_median": float(np.median(det_counts_np)) if n_images else None,
        "detections_per_image_p95": percentile(det_counts_np, 95),
        "detections_per_image_conf_ge_mean": {
            str(t): float(np.mean(dets_per_img_conf_ge[str(t)])) if n_images else None for t in thresholds
        },
        "max_conf_any_mean": float(max_conf_np.mean()) if n_images else None,
        "max_conf_any_median": float(np.median(max_conf_np)) if n_images else None,
        "max_conf_any_p95": percentile(max_conf_np, 95),
        "image_rate_dets_ge": {k: (v / n_images if n_images else None) for k, v in has_ge_k.items()},
        "image_rate_any_det_conf_ge": {k: (v / n_images if n_images else None) for k, v in has_any_conf_ge.items()},
        "total_detections_conf_ge": total_dets_conf_ge,
        "predicted_class_histogram": dict(sorted(class_hist.items(), key=lambda kv: kv[1], reverse=True)),
        "config": {
            "yaml": str(yaml_path),
            "img_dir_used": str(get_image_dir_from_yaml(yaml_path)),
            "imgsz": imgsz,
            "conf_min": conf_min,
            "iou": iou,
            "thresholds": thresholds,
            "det_k_list": det_k_list,
            "max_det": max_det,
        },
    }


# ----------------- B) false positives vs GT -----------------
def read_yolo_gt_labels(txt_path: Path):
    if not txt_path.exists():
        return []
    lines = txt_path.read_text(encoding="utf-8", errors="ignore").strip().splitlines()
    out = []
    for ln in lines:
        parts = ln.strip().split()
        if len(parts) < 5:
            continue
        try:
            c = int(float(parts[0]))
            x, y, w, h = map(float, parts[1:5])
            out.append((c, x, y, w, h))
        except Exception:
            continue
    return out


def xywhn_to_xyxy(x, y, w, h):
    x1 = x - w / 2.0
    y1 = y - h / 2.0
    x2 = x + w / 2.0
    y2 = y + h / 2.0
    return x1, y1, x2, y2


def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)
    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0


def compute_false_positive_metrics(model: YOLO, yaml_path: Path, imgsz: int, conf_min: float, iou: float, thresholds, gt_iou_match: float, max_det: int, device_arg: str):
    img_dir = get_image_dir_from_yaml(yaml_path)
    lab_dir = get_labels_dir_for_images(img_dir)

    fp_per_image = []
    fp_by_thr = {str(t): [] for t in thresholds}

    total_fp = 0
    total_tp = 0
    total_preds = 0
    total_gt = 0

    results_iter = model.predict(
        source=str(img_dir),
        imgsz=imgsz,
        conf=conf_min,
        iou=iou,
        device=device_arg,
        stream=True,
        verbose=False,
        max_det=max_det,
    )

    n_images = 0
    for r in results_iter:
        n_images += 1
        img_name = Path(r.path).name
        stem = Path(img_name).stem
        gt_path = lab_dir / f"{stem}.txt"

        gt = read_yolo_gt_labels(gt_path)
        total_gt += len(gt)

        boxes = r.boxes
        if boxes is None or len(boxes) == 0:
            fp_per_image.append(0)
            for t in thresholds:
                fp_by_thr[str(t)].append(0)
            if n_images == 1 or (n_images % 10 == 0):
                print(f"[B] {yaml_path.name}: processed {n_images} images", flush=True)
            continue

        pred_cls = boxes.cls.detach().cpu().numpy().astype(int)
        pred_xywhn = boxes.xywhn.detach().cpu().numpy().astype(np.float32)
        pred_conf = boxes.conf.detach().cpu().numpy().astype(np.float32)

        preds = []
        for i in range(len(pred_cls)):
            c = int(pred_cls[i])
            x, y, w, h = pred_xywhn[i].tolist()
            preds.append((c, xywhn_to_xyxy(x, y, w, h), float(pred_conf[i])))

        total_preds += len(preds)

        gt_by_class = {}
        for (c, x, y, w, h) in gt:
            gt_by_class.setdefault(int(c), []).append(xywhn_to_xyxy(x, y, w, h))

        def count_fp_at(thr):
            pf = [(c, box, conf) for (c, box, conf) in preds if conf >= thr]
            pf.sort(key=lambda z: z[2], reverse=True)

            tp = 0
            fp = 0
            remaining = {c: list(lst) for c, lst in gt_by_class.items()}

            for c, pbox, _ in pf:
                best_iou = 0.0
                best_j = None
                gtl = remaining.get(c, [])
                for j, gbox in enumerate(gtl):
                    iouv = iou_xyxy(pbox, gbox)
                    if iouv > best_iou:
                        best_iou = iouv
                        best_j = j
                if best_j is not None and best_iou >= gt_iou_match:
                    tp += 1
                    gtl.pop(best_j)
                    remaining[c] = gtl
                else:
                    fp += 1
            return tp, fp, len(pf)

        tp0, fp0, _ = count_fp_at(conf_min)
        total_tp += tp0
        total_fp += fp0
        fp_per_image.append(fp0)

        for t in thresholds:
            _, fpt, _ = count_fp_at(t)
            fp_by_thr[str(t)].append(fpt)

        if n_images == 1 or (n_images % 10 == 0):
            print(f"[B] {yaml_path.name}: processed {n_images} images", flush=True)

    fp_arr = np.asarray(fp_per_image, dtype=np.float32)

    return {
        "images_eval": int(n_images),
        "gt_total_boxes": int(total_gt),
        "pred_total_boxes_conf_ge_conf_min": int(total_preds),
        "tp_total_conf_ge_conf_min": int(total_tp),
        "fp_total_conf_ge_conf_min": int(total_fp),
        "fp_per_image_mean_conf_min": float(fp_arr.mean()) if n_images else None,
        "fp_per_image_median_conf_min": float(np.median(fp_arr)) if n_images else None,
        "fp_per_image_p95_conf_min": percentile(fp_arr, 95),
        "fp_per_image_conf_ge": {str(t): float(np.mean(fp_by_thr[str(t)])) if n_images else None for t in thresholds},
        "config": {
            "yaml": str(yaml_path),
            "img_dir_used": str(img_dir),
            "labels_dir_used": str(lab_dir),
            "imgsz": imgsz,
            "conf_min": conf_min,
            "iou": iou,
            "thresholds": thresholds,
            "gt_iou_match": gt_iou_match,
            "max_det": max_det,
        },
    }


def delta_numeric(a, b):
    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        return b - a
    return None


def dict_delta(a, b):
    keys = sorted(set(a.keys()) | set(b.keys()), key=lambda x: float(x))
    return {k: delta_numeric(a.get(k), b.get(k)) for k in keys}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out_root", required=True)
    ap.add_argument("--run_name", required=True)
    ap.add_argument("--model", required=True)

    ap.add_argument("--clean_root", required=True)
    ap.add_argument("--patched_root", required=True)

    ap.add_argument("--split", default="test")
    ap.add_argument("--imgsz", type=int, default=640)
    ap.add_argument("--conf_min", type=float, default=0.001)
    ap.add_argument("--iou", type=float, default=0.6)
    ap.add_argument("--thresholds", default="0.3,0.5,0.7")
    ap.add_argument("--gt_iou_match", type=float, default=0.5)
    ap.add_argument("--det_k", default="1,5,10,20,50")
    ap.add_argument("--max_det", type=int, default=1000)
    ap.add_argument("--device", default="cpu")
    args = ap.parse_args()

    thresholds = [float(x.strip()) for x in args.thresholds.split(",") if x.strip()]
    det_k_list = [int(x.strip()) for x in args.det_k.split(",") if x.strip()]

    out_root = Path(args.out_root)
    out_root.mkdir(parents=True, exist_ok=True)
    run_dir = out_root / args.run_name
    safe_mkdir(run_dir)

    yamls_dir = safe_mkdir(run_dir / "yamls")

    print(f"RUN DIR: {run_dir}", flush=True)
    print(f"MODEL: {args.model}", flush=True)
    print(f"IMG_SIZE: {args.imgsz}", flush=True)
    print(f"SPLIT: {args.split}", flush=True)
    print(f"MAX_DET: {args.max_det}", flush=True)

    model = YOLO(args.model)
    names = model_names_as_list(model)

    clean_img_dir = Path(args.clean_root) / "images" / args.split
    clean_lab_dir = Path(args.clean_root) / "labels" / args.split
    patched_img_dir = Path(args.patched_root) / "images" / args.split
    patched_lab_dir = Path(args.patched_root) / "labels" / args.split

    if not clean_img_dir.exists():
        raise FileNotFoundError(f"Missing clean images: {clean_img_dir}")
    if not clean_lab_dir.exists():
        raise FileNotFoundError(f"Missing clean labels: {clean_lab_dir}")
    if not patched_img_dir.exists():
        raise FileNotFoundError(f"Missing patched images: {patched_img_dir}")
    if not patched_lab_dir.exists():
        raise FileNotFoundError(f"Missing patched labels: {patched_lab_dir}")

    clean_yaml = yamls_dir / "clean.yaml"
    patched_yaml = yamls_dir / "patched.yaml"
    write_yaml(clean_yaml, Path(args.clean_root), f"images/{args.split}", names)
    write_yaml(patched_yaml, Path(args.patched_root), f"images/{args.split}", names)

    clean_val_log = run_dir / "clean_val.log"
    patched_val_log = run_dir / "patched_val.log"

    cmd_base = [
        "yolo", "val",
        f"model={args.model}",
        f"imgsz={args.imgsz}",
        f"conf={args.conf_min}",
        f"iou={args.iou}",
        f"max_det={args.max_det}",
        f"device={args.device}",
        "split=test",
        "batch=1",
        "workers=0",
        f"project={run_dir}",
    ]

    rc = run(cmd_base + [f"data={clean_yaml}", "name=clean"], clean_val_log)
    if rc != 0:
        raise SystemExit(f"yolo val clean failed (rc={rc}) see {clean_val_log}")

    rc = run(cmd_base + [f"data={patched_yaml}", "name=patched"], patched_val_log)
    if rc != 0:
        raise SystemExit(f"yolo val patched failed (rc={rc}) see {patched_val_log}")

    clean_val = parse_yolo_val_metrics_from_log(clean_val_log)
    patched_val = parse_yolo_val_metrics_from_log(patched_val_log)

    clean_A = compute_detection_density_metrics(
        model, clean_yaml, args.imgsz, args.conf_min, args.iou, thresholds, det_k_list, args.max_det, args.device
    )
    patched_A = compute_detection_density_metrics(
        model, patched_yaml, args.imgsz, args.conf_min, args.iou, thresholds, det_k_list, args.max_det, args.device
    )

    clean_B = compute_false_positive_metrics(
        model, clean_yaml, args.imgsz, args.conf_min, args.iou, thresholds, args.gt_iou_match, args.max_det, args.device
    )
    patched_B = compute_false_positive_metrics(
        model, patched_yaml, args.imgsz, args.conf_min, args.iou, thresholds, args.gt_iou_match, args.max_det, args.device
    )

    delta = {
        "yolo_val": {
            "precision": delta_numeric(clean_val.get("precision"), patched_val.get("precision")),
            "recall": delta_numeric(clean_val.get("recall"), patched_val.get("recall")),
            "mAP50": delta_numeric(clean_val.get("mAP50"), patched_val.get("mAP50")),
            "mAP50-95": delta_numeric(clean_val.get("mAP50-95"), patched_val.get("mAP50-95")),
        },
        "A": {
            "detections_per_image_mean": delta_numeric(clean_A.get("detections_per_image_mean"), patched_A.get("detections_per_image_mean")),
            "detections_total_model": delta_numeric(clean_A.get("detections_total_model"), patched_A.get("detections_total_model")),
            "detections_per_image_conf_ge_mean": dict_delta(
                clean_A.get("detections_per_image_conf_ge_mean", {}),
                patched_A.get("detections_per_image_conf_ge_mean", {})
            ),
        },
        "B": {
            "fp_per_image_mean_conf_min": delta_numeric(clean_B.get("fp_per_image_mean_conf_min"), patched_B.get("fp_per_image_mean_conf_min")),
            "fp_total_conf_ge_conf_min": delta_numeric(clean_B.get("fp_total_conf_ge_conf_min"), patched_B.get("fp_total_conf_ge_conf_min")),
            "fp_per_image_conf_ge": dict_delta(
                clean_B.get("fp_per_image_conf_ge", {}),
                patched_B.get("fp_per_image_conf_ge", {})
            ),
        },
    }

    combined = {
        "run_dir": str(run_dir),
        "model": args.model,
        "imgsz": args.imgsz,
        "split": args.split,
        "max_det": args.max_det,
        "clean": {"yaml": str(clean_yaml), "yolo_val": clean_val, "A": clean_A, "B": clean_B},
        "patched": {"yaml": str(patched_yaml), "yolo_val": patched_val, "A": patched_A, "B": patched_B},
        "delta_patched_minus_clean": delta,
    }

    summary = {
        "run_dir": str(run_dir),
        "model": args.model,
        "imgsz": args.imgsz,
        "split": args.split,
        "max_det": args.max_det,
        "clean_mAP50": clean_val.get("mAP50"),
        "patched_mAP50": patched_val.get("mAP50"),
        "delta_mAP50": delta["yolo_val"]["mAP50"],
        "clean_mAP50_95": clean_val.get("mAP50-95"),
        "patched_mAP50_95": patched_val.get("mAP50-95"),
        "delta_mAP50_95": delta["yolo_val"]["mAP50-95"],
        "clean_det_mean": clean_A.get("detections_per_image_mean"),
        "patched_det_mean": patched_A.get("detections_per_image_mean"),
        "delta_det_mean": delta["A"]["detections_per_image_mean"],
        "clean_fp_mean": clean_B.get("fp_per_image_mean_conf_min"),
        "patched_fp_mean": patched_B.get("fp_per_image_mean_conf_min"),
        "delta_fp_mean": delta["B"]["fp_per_image_mean_conf_min"],
        "clean_det_per_img_conf_ge": clean_A.get("detections_per_image_conf_ge_mean", {}),
        "patched_det_per_img_conf_ge": patched_A.get("detections_per_image_conf_ge_mean", {}),
        "delta_det_per_img_conf_ge": delta["A"]["detections_per_image_conf_ge_mean"],
        "clean_fp_per_img_conf_ge": clean_B.get("fp_per_image_conf_ge", {}),
        "patched_fp_per_img_conf_ge": patched_B.get("fp_per_image_conf_ge", {}),
        "delta_fp_per_img_conf_ge": delta["B"]["fp_per_image_conf_ge"],
    }

    out_combined = run_dir / "combined_metrics.json"
    out_summary = run_dir / "summary.json"
    out_combined.write_text(json.dumps(combined, indent=2), encoding="utf-8")
    out_summary.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print("\nWROTE:", out_combined, flush=True)
    print("WROTE:", out_summary, flush=True)
    print("\nSUMMARY:\n", json.dumps(summary, indent=2), flush=True)


if __name__ == "__main__":
    main()

In [ ]:
# =========================
# 7) Run clean vs patched evaluation
# =========================
import subprocess
import sys
import json
from pathlib import Path

EVAL_SCRIPT = Path("/content/evaluation_ab_realworld.py")

assert EVAL_SCRIPT.exists(), f"Missing eval script: {EVAL_SCRIPT}"
assert CLEAN_EVAL_ROOT.exists(), f"Missing clean eval root: {CLEAN_EVAL_ROOT}"
assert PATCHED_EVAL_ROOT.exists(), f"Missing patched eval root: {PATCHED_EVAL_ROOT}"

EVAL_OUT_ROOT = OUT_ROOT / "eval_ab_runs"
RUN_NAME = f"nova_realworld_clean_vs_patched_{SPLIT}"

cmd = [
    sys.executable, str(EVAL_SCRIPT),
    "--model", str(MODEL_PATH),
    "--clean_root", str(CLEAN_EVAL_ROOT),
    "--patched_root", str(PATCHED_EVAL_ROOT),
    "--out_root", str(EVAL_OUT_ROOT),
    "--run_name", RUN_NAME,
    "--split", SPLIT,
    "--imgsz", str(IMGSZ),
    "--conf_min", str(CONF_MIN),
    "--iou", str(IOU),
    "--thresholds", THRESHOLDS,
    "--gt_iou_match", str(GT_IOU_MATCH),
    "--det_k", DET_K,
    "--max_det", str(MAX_DET),
    "--device", DEVICE,
]

print("[RUNNING]")
print(" ".join(cmd))

proc = subprocess.run(cmd, text=True, capture_output=True)

print(proc.stdout)

if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f"Evaluation failed with return code {proc.returncode}")

run_dir = EVAL_OUT_ROOT / RUN_NAME
combined_path = run_dir / "combined_metrics.json"
summary_path = run_dir / "summary.json"

assert combined_path.exists(), f"Missing: {combined_path}"
assert summary_path.exists(), f"Missing: {summary_path}"

summary = json.loads(summary_path.read_text(encoding="utf-8"))
combined = json.loads(combined_path.read_text(encoding="utf-8"))

print("\n=== SUMMARY ===")
print(json.dumps(summary, indent=2))

print("\nSaved outputs:")
print("Run dir:", run_dir)
print("Combined metrics:", combined_path)
print("Summary:", summary_path)

In [ ]:
# =========================
# 8) Display summary table + plots
# =========================
summary_rows = [
    {
        "metric": "mAP50",
        "clean": summary.get("clean_mAP50"),
        "patched": summary.get("patched_mAP50"),
        "delta_patched_minus_clean": summary.get("delta_mAP50"),
    },
    {
        "metric": "mAP50-95",
        "clean": summary.get("clean_mAP50_95"),
        "patched": summary.get("patched_mAP50_95"),
        "delta_patched_minus_clean": summary.get("delta_mAP50_95"),
    },
    {
        "metric": "detections/image mean",
        "clean": summary.get("clean_det_mean"),
        "patched": summary.get("patched_det_mean"),
        "delta_patched_minus_clean": summary.get("delta_det_mean"),
    },
    {
        "metric": "false positives/image mean",
        "clean": summary.get("clean_fp_mean"),
        "patched": summary.get("patched_fp_mean"),
        "delta_patched_minus_clean": summary.get("delta_fp_mean"),
    },
]

summary_df = pd.DataFrame(summary_rows)
summary_csv = run_dir / "summary_table.csv"
summary_df.to_csv(summary_csv, index=False)

display(summary_df)
print("Saved summary CSV:", summary_csv)

# Bar plot: clean vs patched for key metrics
plot_dir = run_dir / "plots"
plot_dir.mkdir(parents=True, exist_ok=True)

for metric in ["mAP50", "mAP50-95", "detections/image mean", "false positives/image mean"]:
    row = summary_df[summary_df["metric"] == metric].iloc[0]
    values = [row["clean"], row["patched"]]

    plt.figure(figsize=(5, 4))
    plt.bar(["clean", "patched"], values)
    plt.title(metric)
    plt.ylabel(metric)
    plt.tight_layout()

    out_path = plot_dir / f"{metric.replace('/', '_').replace(' ', '_')}.png"
    plt.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.show()

print("Saved plots:", plot_dir)

In [ ]:
# =========================
# 9) Show clean vs patched detections
# =========================
import random
from PIL import Image

EXAMPLE_OUT_DIR = OUT_ROOT / "clean_vs_patched_detection_examples"
EXAMPLE_OUT_DIR.mkdir(parents=True, exist_ok=True)

NUM_EXAMPLES = 6
RANDOM_EXAMPLES = False
PRED_CONF = 0.25
PRED_IOU = 0.7

clean_paths = find_images(clean_img_out)
assert len(clean_paths) > 0, f"No clean images found in {clean_img_out}"

if RANDOM_EXAMPLES:
    selected_clean = random.sample(clean_paths, k=min(NUM_EXAMPLES, len(clean_paths)))
else:
    selected_clean = clean_paths[:NUM_EXAMPLES]

print(f"Showing {len(selected_clean)} examples")
print("Saving comparisons to:", EXAMPLE_OUT_DIR)

for clean_path in selected_clean:
    patched_path = patched_img_out / clean_path.name

    if not patched_path.exists():
        print("Skipping missing patched image:", patched_path)
        continue

    clean_result = model(str(clean_path), imgsz=IMGSZ, conf=PRED_CONF, iou=PRED_IOU, device=DEVICE, verbose=False)[0]
    patched_result = model(str(patched_path), imgsz=IMGSZ, conf=PRED_CONF, iou=PRED_IOU, device=DEVICE, verbose=False)[0]

    # Ultralytics result.plot() returns BGR
    clean_rgb = cv2.cvtColor(clean_result.plot(), cv2.COLOR_BGR2RGB)
    patched_rgb = cv2.cvtColor(patched_result.plot(), cv2.COLOR_BGR2RGB)

    clean_det_path = EXAMPLE_OUT_DIR / f"{clean_path.stem}_clean_det.png"
    patched_det_path = EXAMPLE_OUT_DIR / f"{clean_path.stem}_patched_det.png"
    comparison_path = EXAMPLE_OUT_DIR / f"{clean_path.stem}_clean_vs_patched.png"

    Image.fromarray(clean_rgb).save(clean_det_path)
    Image.fromarray(patched_rgb).save(patched_det_path)

    plt.figure(figsize=(16, 7))

    plt.subplot(1, 2, 1)
    plt.imshow(clean_rgb)
    plt.title(f"Clean detections\\n{clean_path.name}")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(patched_rgb)
    plt.title(f"Patched detections\\n{patched_path.name}")
    plt.axis("off")

    plt.tight_layout()
    plt.savefig(comparison_path, dpi=200, bbox_inches="tight")
    plt.show()

    print("Saved:", comparison_path)

In [ ]:
# =========================
# 10) Save/show the exact 640 preview for example patched images
# =========================
PREVIEW_640_DIR = OUT_ROOT / "model_640_previews"
PREVIEW_640_DIR.mkdir(parents=True, exist_ok=True)

for clean_path in selected_clean[:min(3, len(selected_clean))]:
    patched_path = patched_img_out / clean_path.name

    clean_img = Image.open(clean_path).convert("RGB")
    patched_img = Image.open(patched_path).convert("RGB")

    clean_640 = letterbox_pil(clean_img, IMGSZ)
    patched_640 = letterbox_pil(patched_img, IMGSZ)

    clean_640_path = PREVIEW_640_DIR / f"{clean_path.stem}_clean_640.png"
    patched_640_path = PREVIEW_640_DIR / f"{clean_path.stem}_patched_640.png"

    save_rgb_image(clean_640, clean_640_path)
    save_rgb_image(patched_640, patched_640_path)

    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.imshow(clean_640)
    plt.title("Clean 640 letterbox preview")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(patched_640)
    plt.title("Patched 640 letterbox preview")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

print("Saved 640 previews:", PREVIEW_640_DIR)

## What to change for live matching

The most important variable is the apparent patch size in the **1920×1200 camera frame**:

```python
PATCH_CAMERA_W = 80
PATCH_CAMERA_H = 80
```

With 1920×1200 → YOLO 640 letterbox, that becomes roughly:

```text
80 / 3 ≈ 27 pixels inside YOLO input
```

If you want the patch to appear as 80×80 inside YOLO input, set:

```python
TARGET_PATCH_SIZE_640 = 80
```

That will make the camera-space patch about 240×240.

So:

```text
PATCH_CAMERA_W/H = 80    -> realistic small live patch, YOLO sees ≈27×27
TARGET_PATCH_SIZE_640=80 -> bigger physical patch, YOLO sees ≈80×80
```